In [ ]:
import cv2
from pathlib import Path

# --- Cargar clasificadores Haar ---

# Usar la ruta integrada que provee OpenCV. 
# No anda en la versión 5.0.0.93, es un bug que se está arreglando, mientras tanto lo usaré localmente.
# face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
# eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_eye.xml")
# smile_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_smile.xml")
    
# Cargar clasificadores Haar desde archivos locales
cascade_dir = Path("haar_classifiers")

face_cascade_path = cascade_dir / "haarcascade_frontalface_default.xml"
eye_cascade_path = cascade_dir / "haarcascade_eye.xml"
smile_cascade_path = cascade_dir / "haarcascade_smile.xml"

# Cargar clasificadores
face_cascade = cv2.CascadeClassifier(str(face_cascade_path))
eye_cascade = cv2.CascadeClassifier(str(eye_cascade_path))
smile_cascade = cv2.CascadeClassifier(str(smile_cascade_path))

# Verificación explícita antes de procesar la imagen
if face_cascade.empty():
    raise FileNotFoundError("No se pudo cargar el clasificador de caras. Verifica la ruta.")
if eye_cascade.empty():
    raise FileNotFoundError("No se pudo cargar el clasificador de ojos. Verifica la ruta.")
if smile_cascade.empty():
    raise FileNotFoundError("No se pudo cargar el clasificador de sonrisas. Verifica la ruta.")

# --- Capturar desde webcam ---
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Convertir a escala de grises
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detectar caras
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    estado = "neutral"

    for (x, y, w, h) in faces:
        # Dibujar rectángulo sobre la cara
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        roi_gray = gray[y:y+h, x:x+w]
        roi_color = frame[y:y+h, x:x+w]

        # Detectar ojos
        eyes = eye_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=10, minSize=(30, 30))
        for (ex, ey, ew, eh) in eyes:
            cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (0, 255, 0), 2)

        # Detectar sonrisa
        smiles = smile_cascade.detectMultiScale(roi_gray, scaleFactor=1.7, minNeighbors=22, minSize=(25, 25))
        for (sx, sy, sw, sh) in smiles:
            cv2.rectangle(roi_color, (sx, sy), (sx+sw, sy+sh), (0, 0, 255), 2)

        # Clasificación simple según detecciones
        if len(smiles) > 0:
            estado = "feliz"
        elif len(eyes) == 0:
            estado = "dormido"
        else:
            estado = "neutral"

    # Mostrar texto en pantalla
    cv2.putText(frame, f"Estado: {estado}", (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)

    # Mostrar frame
    cv2.imshow('Detector Haar', frame)

    if cv2.waitKey(1) & 0xFF == 27:  # tecla ESC
        break

cap.release()
cv2.destroyAllWindows()


KeyboardInterrupt: 